# dictionaries
Dictionaries are Python’s key-value workhorse. In engineering systems they model configs, JSON objects, caches, indexes, counters, routing tables, and normalized records. Their power is fast keyed access, but the real skill is designing clear keys and safe update behavior.

## 1. Problem First
Why does this matter in real systems?
- Most API payloads naturally become dictionaries.
- Config objects are often nested mappings with optional keys.
- Indexing by key is far faster than repeated scans through lists.

In [ ]:
record = {"service": "api", "status": "ok", "latency_ms": 120}
print(record["service"])
print(record["latency_ms"])

## 2. Minimal Concept
Core syntax:
- Create with `{key: value}`
- Access with `dict[key]` or `dict.get(key)`
- Add or update with assignment
- Keys must be hashable

## 3. Mental Model
How Python actually behaves
A dictionary is a hash-table-backed mapping from keys to values. Keys are looked up by hash and equality, not by position. Modern Python preserves insertion order, but the main semantic value of a dict is keyed access, not sequence behavior.

In [ ]:
config = {"host": "api.internal", "port": 8080}
config["debug"] = False
print(config)
print(config.get("timeout", 30))

## 4. Internal Mechanics
Dictionary lookup is based on hashable keys, which is why mutable values like lists cannot be used as keys. Key access can raise `KeyError` if you assume presence incorrectly. The choice between `[]` and `get()` changes failure behavior.

In [ ]:
import dis

def read_status(payload):
    return payload["status"]

dis.dis(read_status)
try:
    print(read_status({}))
except KeyError as exc:
    print(type(exc).__name__)
try:
    bad = {[1, 2]: "x"}
except TypeError as exc:
    print(type(exc).__name__)

## 5. Edge Cases
Where it breaks:
- Optional keys can trigger `KeyError` if accessed directly.
- Shallow copying a dict does not isolate nested mutable values.
- Overwriting an existing key may silently discard previous data.
- Using unstable or unclear keys leads to maintenance bugs.

In [ ]:
base = {"meta": {"retries": 3}}
shallow = base.copy()
shallow["meta"]["retries"] = 10
print(base)
print(shallow)

## 6. Performance Thinking
- Average lookup, insert, and delete are O(1).
- Iterating all key-value pairs is O(n).
- A dict is usually the right tool when repeated lookup by identifier matters.
- Memory overhead is higher than a plain list because hashing infrastructure exists.

## 7. Real Use Case
A service registry often uses a dictionary keyed by service name or ID to find config and health information quickly.

In [ ]:
registry = {
    "api": {"host": "api.internal", "healthy": True},
    "worker": {"host": "worker.internal", "healthy": False}
}
print(registry["api"]["host"])
print(registry.get("scheduler", {"healthy": None}))

## 8. Anti-Pattern
What beginners do wrong:
- Use a list of dictionaries when direct keyed lookup is the actual need.
- Reach for direct indexing on optional fields.
- Overload one dictionary with too many unrelated concerns.

In [ ]:
users = [
    {"id": 1, "name": "a"},
    {"id": 2, "name": "b"}
]
target_id = 2
for user in users:
    if user["id"] == target_id:
        print(user)


## 9. Interview Signals
Questions FAANG asks:
- Why are dictionary lookups usually O(1)?
- What makes a key hashable?
- When should `get()` be used instead of direct indexing?
- How do shallow copies of dicts behave with nested values?

## 10. Exercise (Non-trivial)
Design an in-memory index for API records keyed by `id`. Decide how to handle duplicates, optional fields, nested metadata, and missing lookups. Explain why a dictionary is a better fit than scanning a list for each request.

In [ ]:
def index_records(records):
    indexed = {}
    for record in records:
        indexed[record["id"]] = record
    return indexed

# Task:
# 1. Decide what to do with duplicate ids.
# 2. Add safe lookup behavior.
# 3. Explain complexity compared to list scanning.